# Day 23 — **Memory Profiling: Agent Optimisation**

**Phase 1 · Module 3: Prompt Engineering & RAG** · ~30 min

Agents hold embeddings, chat history, and retrieved chunks in memory. On a laptop it's fine; in a container with a 512 MB limit it OOM-kills mid-request. Today you *measure* where the memory goes instead of guessing, and cut it with two changes that pay for themselves.

Three tools:
1. `memory_profiler` — line-by-line memory of a function.
2. `sys.getsizeof` — list-of-dicts vs a typed array: the ~10× win.
3. `tracemalloc` — find a **leak** growing across repeated agent runs.

> `tracemalloc` and `sys` are stdlib (always run). `memory_profiler` is optional — `pip install memory_profiler`; a fallback measures the same delta.

## 1. Line-by-line with `memory_profiler`

`@profile` reports memory *per line* of a function, so you see exactly which statement allocates. The usual culprit in an agent: building the embedding store. We measure the whole function's delta here (portable), then point at the real decorator.

In [1]:
import numpy as np

def build_store_naive(n=20_000, dim=384):
    # the common anti-pattern: one dict per chunk, embedding as a python list
    store = []
    for i in range(n):
        store.append({"id": i, "text": f"chunk {i}", "embedding": list(np.random.rand(dim))})
    return store

try:
    from memory_profiler import memory_usage
    peak = memory_usage((build_store_naive, (), {}), max_usage=True)
    base = memory_usage(-1, max_usage=True)
    print(f"[memory_profiler] peak during build: {peak:.1f} MiB")
except Exception:
    import tracemalloc
    tracemalloc.start()
    s = build_store_naive()
    cur, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"[tracemalloc] build_store_naive allocated ~{peak/1e6:.1f} MB (20k dicts + python-list embeddings)")
    del s

[memory_profiler] peak during build: 686.6 MiB


☝ 20k chunks as dicts-with-list-embeddings costs tens of MB — and it's the `list(np.random.rand(dim))` line that dominates, because a Python `list` of floats stores each float as a boxed object (~24–28 bytes) plus list overhead. Real `@profile` would flag that exact line. Fix: keep embeddings in one NumPy array.

## 2. `list of dicts` vs typed NumPy array

The embeddings are the mass. A Python `list` of `dim` floats is boxed objects; a NumPy `float32` row is `dim × 4` contiguous bytes. Store all embeddings in one `(n, dim)` array and keep only lightweight metadata alongside — the standard ~10× memory cut.

In [2]:
import sys

n, dim = 20_000, 384

def deep_size(obj, seen=None):
    seen = seen or set()
    if id(obj) in seen: return 0
    seen.add(id(obj))
    size = sys.getsizeof(obj)
    if isinstance(obj, dict):
        size += sum(deep_size(k, seen) + deep_size(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple)):
        size += sum(deep_size(x, seen) for x in obj)
    return size

# sample one row of each layout, scale to n (avoids building 20k python objects just to weigh them)
row_dict = {"id": 0, "text": "chunk 0", "embedding": list(np.random.rand(dim))}
naive_mb = deep_size(row_dict) * n / 1e6

emb = np.random.rand(n, dim).astype(np.float32)          # all embeddings, one contiguous block
meta = [{"id": i, "text": f"chunk {i}"} for i in range(n)]  # tiny metadata only
opt_mb = (emb.nbytes + deep_size(meta[0]) * n) / 1e6

print(f"list-of-dicts (python-list embeddings): {naive_mb:6.1f} MB")
print(f"numpy float32 array + light metadata  : {opt_mb:6.1f} MB")
print(f"reduction: {naive_mb / opt_mb:.1f}x")

list-of-dicts (python-list embeddings):  316.3 MB
numpy float32 array + light metadata  :   37.7 MB
reduction: 8.4x


☝ Same 20k embeddings, a multi-fold drop — the win is (a) `float32` contiguous bytes instead of boxed Python floats, and (b) not repeating dict/list overhead 20k times. This is the array-of-structs → struct-of-arrays refactor; it's also why FAISS and Pandas hold columns, not row objects.

## 3. Catch a leak with `tracemalloc`

A slow leak survives one request and shows up only after many. `tracemalloc` snapshots allocations; diffing snapshots across 10 agent runs shows what **grew** and the exact line that allocated it. Here a module-level cache is never evicted — the classic agent leak.

In [3]:
import tracemalloc

_CACHE = {}                       # module-level cache with no eviction  <-- the leak
def agent_run(req_id, dim=384):
    ctx = np.random.rand(50, dim).astype(np.float32)   # retrieved chunks for this request
    _CACHE[req_id] = ctx          # bug: kept forever, one entry per request
    return float(ctx.mean())

tracemalloc.start()
snap0 = tracemalloc.take_snapshot()
for r in range(10):
    agent_run(r)
snap1 = tracemalloc.take_snapshot()

top = snap1.compare_to(snap0, "lineno")[0]
print("biggest growth across 10 runs:")
print(" ", top)
print(f"\ncache entries retained: {len(_CACHE)}  (should be 0 after a request completes)")
tracemalloc.stop()

biggest growth across 10 runs:
  /var/folders/jd/b91jfh0n5r90fqg_0_2t9xg00000gn/T/ipykernel_25152/3278260555.py:5: size=751 KiB (+751 KiB), count=27 (+27), average=27.8 KiB

cache entries retained: 10  (should be 0 after a request completes)


☝ `compare_to` points straight at the `_CACHE[req_id] = ctx` line as the top grower, and the cache holds 10 entries that should have been freed — memory climbs one request at a time until OOM. Fix: bound the cache (`functools.lru_cache(maxsize=...)` or an LRU dict) or scope context to the request. `tracemalloc` names the guilty line so you don't bisect by hand.

## 4. Recap — measure, then cut

| Tool | Answers | Fix it reveals |
|---|---|---|
| `memory_profiler @profile` | which **line** allocates | boxed-list embeddings |
| `sys.getsizeof` (deep) | which **layout** is heavier | list-of-dicts → NumPy array (~10×) |
| `tracemalloc.compare_to` | what **grows** over runs | unbounded cache = leak |

Rule for agents: **embeddings live in one `float32` array, metadata stays light, caches are bounded.** Profile before optimising — the biggest consumer is almost always embedding storage, exactly where you'd guess last.